## 1. Imports et configuration

In [ ]:
import os, re, json, math, random, hashlib, unicodedata, platform, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)

STEP2_DIR = Path("outputs/step2_index")          # produit par le notebook de l'étape 2
SEED_PATH = Path("train_seed_b1.jsonl")
TEST_SET_PATH = Path("test_set_b1.jsonl")
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step3_dataset"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_FRACTION = 0.20            # part des questions-sources réservée à la validation (split PAR QUESTION, jamais par exemple)
N_VARIANTS_TRAIN = 2           # nombre de contextes différents générés par question d'entraînement
MISS_RATE = 0.15               # part des questions « avec réponse » dont on génère AUSSI une variante « retrieval manqué » (réponse absente du contexte -> abstention)
MISS_ANSWER_OVERLAP = 0.5       # une variante « miss » est écartée si ≥ 50 % du contenu de la réponse est encore dans un passage du contexte (le guide répète certains faits)
ABSTAIN_MARKER = "INFORMATION ABSENTE"          # identique à l'étape 1 (les métriques d'abstention en dépendent)
ABSTAIN_ANSWER = ABSTAIN_MARKER + " : les passages fournis ne contiennent pas cette information."
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

random.seed(SEED); np.random.seed(SEED)
rng = random.Random(SEED)
print("python", platform.python_version())

python 3.13.16


## 2. Utilitaires (identiques à l'étape 2 : normalisation, BM25, citation)

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", s.lower().replace("’", "'"))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s  ]+", " ", s).strip()

STOPWORDS = set("le la les l un une des de du d et ou a au aux en dans par pour sur sous avec sans que qui quoi est sont etre ce cet cette ces se sa son ses "
                "leur leurs ne pas plus il ils elle elles on y c s n qu j m t si comme mais donc car faut faire peut doit quel quelle quels quelles".split())
def tokenize(text):
    toks = re.findall(r"[a-z0-9]+", norm(text))
    return [t[:-1] if len(t) > 3 and t.endswith("s") else t for t in toks if t not in STOPWORDS]

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [Counter(d) for d in docs_tokens]
        self.len = np.array([len(d) for d in docs_tokens], dtype=float); self.avg = self.len.mean()
        df = Counter(t for d in self.docs for t in d); n = len(docs_tokens)
        self.idf = {t: math.log(1 + (n - f + 0.5) / (f + 0.5)) for t, f in df.items()}
    def scores(self, q_tokens):
        s = np.zeros(len(self.docs))
        for t in set(q_tokens):
            if t in self.idf:
                tf = np.array([d.get(t, 0) for d in self.docs], dtype=float)
                s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.len / self.avg))
        return s

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

def cite(c):
    p = [x for x in c["pages"] if x >= 1]            # la page de présentation du PDF n'a pas de numéro imprimé
    pages = "page non numérotée" if not p else (f"p. {p[0]}" if len(p) == 1 else f"p. {p[0]}-{p[-1]}")
    where = "Présentation du guide" if c["fiche"] == 0 else f"Fiche {c['fiche']} — {c['fiche_title']}"
    return f"{where} — {c['section_label']}, {pages}"

from functools import lru_cache
@lru_cache(maxsize=None)
def rx(pattern):
    """Regex compilée sur le texte normalisé (mise en cache : le scan de fuite l'appelle des centaines de milliers de fois)."""
    return re.compile(norm(pattern))

def jaccard(a, b):
    a, b = set(tokenize(a)), set(tokenize(b))
    return len(a & b) / max(len(a | b), 1)

## 3. Chargement de l'index (étape 2) et des jeux de données



In [ ]:
index_cfg = json.load(open(STEP2_DIR / "index_config.json", encoding="utf-8"))
chunks = load_jsonl(STEP2_DIR / "chunks.jsonl")
K_CONTEXT = int(index_cfg["k_final"])
assert K_CONTEXT >= 3, f"k_final={K_CONTEXT} : trop petit pour une réponse réglementaire complète — relancer l'étape 2 (règle de complétude rep@k)."
print(f"{len(chunks)} chunks | config étape 2 : chunking={index_cfg['chunking']} embedder={index_cfg['embedder']} méthode={index_cfg['method']} | k={K_CONTEXT}")
chunk_norm = [norm(c["text"]) for c in chunks]
bm25 = BM25([tokenize(c["embed_text"]) for c in chunks])

seed_rows = load_jsonl(SEED_PATH); test_rows = load_jsonl(TEST_SET_PATH); dev_rows = load_jsonl(DEV_SET_PATH)
SEED_SHA = hashlib.sha256(SEED_PATH.read_bytes()).hexdigest(); TEST_SHA = hashlib.sha256(TEST_SET_PATH.read_bytes()).hexdigest()
if index_cfg.get("test_sha256") and index_cfg["test_sha256"] != TEST_SHA:
    print("[!] ATTENTION : le test_set_b1.jsonl actuel n'est pas celui utilisé à l'étape 2 (hash différent). Vérifier qu'il s'agit bien du même fichier.")
print(f"graine : {len(seed_rows)} questions | test : {len(test_rows)} | dev : {len(dev_rows)}")
print(Counter(r["type"] for r in seed_rows))

239 chunks | config étape 2 : chunking={'max_chars': 900, 'overlap_units': 1} embedder=bge-m3 méthode=dense | k=5
graine : 190 questions | test : 50 | dev : 75
Counter({'answerable': 150, 'unanswerable_near': 30, 'unanswerable_off': 10})


## 4. Garde-fous anti-fuite

### 4.1 Questions : aucune question commune ou quasi-identique avec le test ou le dev

In [ ]:
held_out = [("test", r["question"]) for r in test_rows] + [("dev", r["question"]) for r in dev_rows]
violations = []
for r in seed_rows:
    for src, q in held_out:
        if norm(r["question"]) == norm(q) or jaccard(r["question"], q) >= 0.8:
            violations.append((r["id"], src, r["question"], q))
print("Questions d'entraînement identiques/quasi-identiques à test/dev :", violations or "aucune")
assert not violations

Questions d'entraînement identiques/quasi-identiques à test/dev : aucune


### 4.2 Passages : les passages qui contiennent la réponse d'une question de test sont **interdits** dans tout contexte d'entraînement



In [ ]:
def group_rate(alts, chunk_norm):
    return np.mean([any(rx(a).search(t) for a in alts) for t in chunk_norm])

def informative_groups(key_facts, chunk_norm, max_rate=0.15):
    return [alts for alts in key_facts if group_rate(alts, chunk_norm) <= max_rate]

def blocked_chunk_ids(test_rows, chunks, chunk_norm):
    blocked, why = set(), defaultdict(list)
    for r in test_rows:
        if r["category"] != "in_corpus":
            continue
        info = informative_groups(r["key_facts"], chunk_norm)
        specific_single = len(info) == 1 and group_rate(info[0], chunk_norm) <= 0.05
        for j, c in enumerate(chunks):
            rule1 = (c["fiche"] in r["gold_fiches"] and set(c["pages"]) & set(r["gold_pages"])
                     and any(rx(a).search(chunk_norm[j]) for alts in r["key_facts"] for a in alts))
            rule2 = bool(info) and (len(info) >= 2 or specific_single) and \
                    all(any(rx(a).search(chunk_norm[j]) for a in alts) for alts in info)
            if rule1 or rule2:
                blocked.add(c["chunk_id"]); why[c["chunk_id"]].append((r["id"], "zone" if rule1 else "doublon"))
    return blocked, why

BLOCKED, BLOCK_WHY = blocked_chunk_ids(test_rows, chunks, chunk_norm)
usable = [j for j, c in enumerate(chunks) if c["chunk_id"] not in BLOCKED]
n_dup = sum(1 for cid, w in BLOCK_WHY.items() if all(k == "doublon" for _, k in w))
print(f"{len(BLOCKED)} chunks bloqués sur {len(chunks)} ({len(BLOCKED)/len(chunks):.0%}) | dont {n_dup} uniquement par la règle « fait dupliqué ailleurs » | {len(usable)} utilisables pour l'entraînement")
cov = pd.DataFrame({"fiche": [c["fiche"] for c in chunks], "bloqué": [c["chunk_id"] in BLOCKED for c in chunks]})
display(cov.groupby("fiche").bloqué.agg(["sum", "count"]).rename(columns={"sum": "bloqués", "count": "chunks"}).T)

75 chunks bloqués sur 239 (31%) | dont 12 uniquement par la règle « fait dupliqué ailleurs » | 164 utilisables pour l'entraînement


fiche,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
bloqués,1,2,2,3,11,3,2,2,3,5,...,2,3,2,3,2,4,4,0,6,0
chunks,5,14,16,8,16,4,9,9,8,9,...,9,5,5,9,14,8,13,11,12,6


## 5. Format du prompt RAG (source de vérité pour les étapes 3, 4, 5 et 6)


In [ ]:
SYSTEM_PROMPT = (
    "Tu es un assistant réglementaire spécialisé dans la sécurité des données personnelles. Tu réponds en français. "
    "Tu réponds UNIQUEMENT à partir des passages fournis, extraits du « Guide pratique RGPD – Sécurité des données personnelles » de la CNIL. "
    "N'utilise aucune connaissance extérieure aux passages.\n"
    "Règles :\n"
    f"1. Si les passages ne contiennent pas l'information demandée, réponds exactement : {ABSTAIN_ANSWER}\n"
    "2. Sinon, réponds en trois phrases maximum, sans rien ajouter qui ne figure pas dans les passages, "
    "puis termine par une dernière ligne « Sources : [n] » listant uniquement les numéros des passages utilisés (ex. : Sources : [2], [4])."
)
USER_TEMPLATE = "Passages :\n\n{passages}\n\nQuestion : {question}"
PASSAGE_TEMPLATE = "[{n}] ({citation})\n{text}"

def build_messages(question, passages, answer=None):
    """passages : liste de chunks (dict). Retourne la liste de messages chat (avec la réponse si fournie)."""
    ctx = "\n\n".join(PASSAGE_TEMPLATE.format(n=i + 1, citation=cite(c), text=c["text"]) for i, c in enumerate(passages))
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(passages=ctx, question=question)}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": answer})
    return msgs

json.dump({"system": SYSTEM_PROMPT, "user_template": USER_TEMPLATE, "passage_template": PASSAGE_TEMPLATE,
           "abstain_marker": ABSTAIN_MARKER, "abstain_answer": ABSTAIN_ANSWER, "k_context": K_CONTEXT},
          open(OUTPUT_DIR / "rag_prompt.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(build_messages("Exemple de question ?", chunks[:2])[1]["content"][:700])

Passages :

[1] (Présentation du guide — Introduction, page non numérotée)
L'objectif de ce guide est d'accompagner les organismes dans la mise en place de mesures de sécurité pour assurer la protection des données personnelles qu'ils traitent.

Il s'adresse notamment aux délégués à la protection des données (DPD), aux responsables de la sécurité des systèmes d'information (RSSI) et aux informaticiens. Les juristes en protection de la vie privée pourront également y trouver des éléments utiles.

Ce guide est un outil vivant qui est enrichi des pratiques à l'état de l'art et des éléments de doctrine de la Commission nationale de l'informatique et des libertés (CNIL) sur la question de la sécu


## 6. Assemblage des exemples


In [ ]:
def evidence_hits(patterns, idx):
    return [any(rx(p).search(chunk_norm[j]) for p in patterns) for j in idx]

def noise_pool(question, forbid_idx, patterns, n, rng=None, hard=True, pool_mult=3):
    """Chunks BM25 de la question, hors chunks interdits, bloqués, ou contenant la preuve.
    hard=True : les n meilleurs (bruit le plus proche) ; sinon n tirés au hasard parmi les n*pool_mult meilleurs (diversité entre variantes)."""
    order = np.argsort(-bm25.scores(tokenize(question)))
    out = []
    for j in order:
        j = int(j)
        if j in forbid_idx or chunks[j]["chunk_id"] in BLOCKED:
            continue
        if patterns and any(rx(p).search(chunk_norm[j]) for p in patterns):
            continue
        out.append(j)
        if len(out) == n * (1 if hard else pool_mult):
            break
    return out if hard or rng is None else rng.sample(out, min(n, len(out)))

def make_example(row, kind, rng, k=K_CONTEXT, hard=True):
    """kind : 'answer' | 'miss' | 'unanswerable'. Retourne un dict (ou None si l'exemple est impossible)."""
    pats = row["evidence"]
    ev_idx = [j for j in range(len(chunks)) if pats and all(rx(p).search(chunk_norm[j]) for p in pats)] if len(pats) == 1 else \
             sorted({j for p in pats for j in range(len(chunks)) if rx(p).search(chunk_norm[j])})
    if kind == "answer":
        cand = [j for j in ev_idx if chunks[j]["chunk_id"] not in BLOCKED and chunks[j]["fiche"] == row["fiche"]]
        if not cand:
            return None, ("preuve uniquement dans des chunks bloqués/absents" if ev_idx else "preuve introuvable")
        if len(pats) == 1:
            gold = [rng.choice(cand)]
        else:                                                      # plusieurs preuves : un chunk par motif
            gold = []
            for p in pats:
                c_p = [j for j in cand if rx(p).search(chunk_norm[j])]
                if not c_p:
                    return None, "une des preuves n'est dans aucun chunk utilisable"
                gold.append(rng.choice(c_p))
            gold = sorted(set(gold))
        noise = noise_pool(row["question"], set(ev_idx), pats, k - len(gold), rng, hard)
        idx = noise + gold; rng.shuffle(idx)
        positions = [idx.index(g) + 1 for g in gold]
        answer = row["answer"].strip() + "\nSources : " + ", ".join(f"[{p}]" for p in sorted(positions))
        meta = {"gold_chunks": [chunks[g]["chunk_id"] for g in gold], "gold_positions": sorted(positions)}
    else:
        forbid = set(ev_idx)
        idx = noise_pool(row["question"], forbid, pats, k, rng, hard)
        rng.shuffle(idx)
        if kind == "miss":
            a = set(tokenize(row["answer"]))
            if a and max(len(a & set(tokenize(chunks[j]["text"]))) / len(a) for j in idx) >= MISS_ANSWER_OVERLAP:
                return None, "réponse encore présente dans le contexte (fait dupliqué ailleurs) : cible « abstention » fausse"
        answer = ABSTAIN_ANSWER
        meta = {"gold_chunks": [], "gold_positions": []}
    passages = [chunks[j] for j in idx]
    return {"id": row["id"], "kind": kind, "question": row["question"], "fiche": row["fiche"], "evidence": pats,
            "context_chunks": [c["chunk_id"] for c in passages], "answer": answer, **meta,
            "messages": build_messages(row["question"], passages, answer)}, None

### 6.1 Split train / val **par question-source** (stratifié par type)



In [ ]:
by_type = defaultdict(list)
for r in seed_rows:
    by_type["answerable" if r["type"] == "answerable" else "unanswerable"].append(r)
split_of = {}
for t, rows_t in by_type.items():
    ids = [r["id"] for r in rows_t]; rng.shuffle(ids)
    n_val = max(1, round(VAL_FRACTION * len(ids)))
    for i, rid in enumerate(ids):
        split_of[rid] = "val" if i < n_val else "train"
print(Counter((r["type"] == "answerable", split_of[r["id"]]) for r in seed_rows))

Counter({(True, 'train'): 120, (False, 'train'): 32, (True, 'val'): 30, (False, 'val'): 8})


### 6.2 Génération

In [ ]:
examples, dropped, miss_dropped = [], [], []
for row in seed_rows:
    split = split_of[row["id"]]
    n_var = N_VARIANTS_TRAIN if split == "train" else 1
    if row["type"] == "answerable":
        seen = set()
        for v in range(n_var):
            for _ in range(8):                         # on tente d'obtenir des contextes différents d'une variante à l'autre
                ex, why = make_example(row, "answer", rng, hard=(v == 0))
                if ex is None or tuple(sorted(ex["context_chunks"])) not in seen:
                    break
            if ex is None:
                dropped.append((row["id"], row["question"], why)); break
            seen.add(tuple(sorted(ex["context_chunks"]))); ex.update(split=split, variant=v); examples.append(ex)
        else:
            if rng.random() < MISS_RATE:
                ex, why = make_example(row, "miss", rng, hard=rng.random() < 0.5)
                if ex is not None:
                    ex.update(split=split, variant=0); examples.append(ex)
                elif why and "dupliqué" in why:
                    miss_dropped.append(row["id"])
    else:
        for v in range(1 if row["type"] == "unanswerable_off" else n_var):       # hors-sujet : 1 variante suffit (cas faciles, ne pas sur-peser l'abstention)
            ex, why = make_example(row, "unanswerable", rng, hard=(v == 0))
            if ex is None:
                dropped.append((row["id"], row["question"], why)); break
            ex.update(split=split, variant=v); examples.append(ex)

df = pd.DataFrame(examples)
print(f"{len(df)} exemples | par split et type :")
display(df.groupby(["split", "kind"]).size().unstack(fill_value=0))
print(f"variantes « miss » écartées car la réponse reste dans le contexte : {len(miss_dropped)} {miss_dropped}")
print(f"{len(dropped)} question(s) écartée(s) (preuve uniquement dans des chunks bloqués, c'est-à-dire proches des réponses du test). Premières :")
for d in dropped[:10]:
    print("  -", d[0], "|", d[2], "|", d[1])
n_ans_seeds = len({ex["id"] for ex in examples if ex["kind"] == "answer"})
print(f"Questions-sources « avec réponse » conservées : {n_ans_seeds} / {sum(r['type'] == 'answerable' for r in seed_rows)}")
if n_ans_seeds < 100:
    print("[!] Moins de 100 questions-sources utilisables : le blocage anti-fuite retire beaucoup de contenu avec ce chunking (gros chunks = plus de blocage). "
          "Options : (a) accepter, (b) ajouter des questions-sources sur des faits non testés, (c) revenir à des chunks plus petits à l'étape 2.")

254 exemples | par split et type :


kind,answer,miss,unanswerable
split,,,
train,164,7,55
val,16,4,8


variantes « miss » écartées car la réponse reste dans le contexte : 2 ['tr106', 'tr129']
52 question(s) écartée(s) (preuve uniquement dans des chunks bloqués, c'est-à-dire proches des réponses du test). Premières :
  - tr001 | preuve uniquement dans des chunks bloqués/absents | À quels profils s'adresse en priorité ce guide ?
  - tr005 | preuve uniquement dans des chunks bloqués/absents | Comment repérer les évolutions du guide pour adapter son niveau de sécurité ?
  - tr017 | preuve uniquement dans des chunks bloqués/absents | Faut-il une session obligatoire sur la protection des données pour les nouveaux arrivants ?
  - tr019 | preuve uniquement dans des chunks bloqués/absents | Faut-il empêcher l'usage de la fonction « Coller » dans les formulaires de mot de passe ?
  - tr020 | preuve uniquement dans des chunks bloqués/absents | Que recommande le guide pour limiter les attaques par essais successifs sur les comptes des postes de travail ?
  - tr021 | preuve uniquement dans des chunk

## 7. Contrôles de qualité automatiques

In [ ]:
SOURCES_RE = re.compile(r"\nSources : (\[\d+\](?:, \[\d+\])*)$")
NUM_RE = re.compile(r"\b\d[\d\s]*\d\b|\b\d\b|\b[A-Z]{3,}[A-Za-z0-9/]*\b")

def check_example(ex):
    errs = []
    idx_by_id = {c["chunk_id"]: j for j, c in enumerate(chunks)}
    ctx_idx = [idx_by_id[cid] for cid in ex["context_chunks"]]
    pats = ex["evidence"]
    ctx_has_ev = [bool(pats) and any(rx(p).search(chunk_norm[j]) for p in pats) for j in ctx_idx]
    if len(ex["context_chunks"]) != len(set(ex["context_chunks"])): errs.append("passages dupliqués")
    if any(cid in BLOCKED for cid in ex["context_chunks"]): errs.append("passage bloqué (réponse du test) dans le contexte")
    if ex["kind"] == "answer":
        m = SOURCES_RE.search(ex["answer"])
        if not m: return errs + ["ligne Sources absente ou mal formée"]
        cited = [int(x) for x in re.findall(r"\d+", m.group(1))]
        if any(c < 1 or c > len(ctx_idx) for c in cited): return errs + ["numéro de passage invalide"]
        if sorted(cited) != sorted(ex["gold_positions"]): errs.append("citations ≠ position du passage-réponse")
        if not any(ctx_has_ev[c - 1] for c in cited): errs.append("la preuve n'est pas dans les passages cités")
        extra = [i + 1 for i, h in enumerate(ctx_has_ev) if h and (i + 1) not in cited]
        if extra: errs.append(f"preuve aussi dans des passages non cités {extra}")
        body = ex["answer"][:m.start()]
        cited_text = " ".join(chunk_norm[ctx_idx[c - 1]] for c in cited)
        for tok in set(NUM_RE.findall(body)):
            if norm(tok) not in cited_text: errs.append(f"jeton « {tok} » absent des passages cités")
        if len(re.findall(r"[.!?](?:\s|$)", body)) > 3: errs.append("plus de 3 phrases")
    else:
        if ex["answer"] != ABSTAIN_ANSWER: errs.append("réponse ≠ abstention exacte")
        if any(ctx_has_ev): errs.append("la preuve est présente alors que la cible est une abstention")
    return errs

df["errors"] = [check_example(ex) for ex in examples]
bad = df[df.errors.apply(len) > 0]
print(f"{len(bad)} exemple(s) fautif(s) sur {len(df)}")
for _, r in bad.head(15).iterrows():
    print(" -", r["id"], r["kind"], r["errors"])
df_ok = df[df.errors.apply(len) == 0].reset_index(drop=True)
examples_ok = [ex for ex, e in zip(examples, df.errors) if not e]

0 exemple(s) fautif(s) sur 254


## 8. Équilibre du dataset

In [ ]:
tab = df_ok.groupby(["split", "kind"]).size().unstack(fill_value=0)
tab["total"] = tab.sum(axis=1); tab["% abstention"] = ((tab.get("miss", 0) + tab.get("unanswerable", 0)) / tab["total"] * 100).round(0)
display(tab)
tr = df_ok[df_ok.split == "train"]
abst_rate = (tr.kind != "answer").mean()
print(f"Part d'abstentions dans le train : {abst_rate:.0%} (cible 20–40 %)")
assert 0.15 <= abst_rate <= 0.45, "taux d'abstention hors cible : ajuster MISS_RATE / N_VARIANTS_TRAIN"

pos = Counter(p for ps in tr[tr.kind == "answer"].gold_positions for p in ps)
print("Position du passage-réponse (train) :", dict(sorted(pos.items())), "-> doit être à peu près uniforme (sinon le modèle apprend « la réponse est en [1] »)")
fich = Counter(tr[tr.kind == "answer"].fiche)
print("Fiches couvertes (questions avec réponse, train) :", len([f for f in fich if f >= 1]), "/ 25 ; fiches non couvertes :", sorted(set(range(1, 26)) - set(fich)))
seed_type = {r["id"]: r["type"] for r in seed_rows}
print("Exemples hors guide (train) par type :", dict(Counter(seed_type[i] for i in tr[tr.kind == "unanswerable"].id)))

kind,answer,miss,unanswerable,total,% abstention
split,,,,,
train,164,7,55,226,27.0
val,16,4,8,28,43.0


Part d'abstentions dans le train : 27% (cible 20–40 %)
Position du passage-réponse (train) : {1: 33, 2: 31, 3: 33, 4: 31, 5: 36} -> doit être à peu près uniforme (sinon le modèle apprend « la réponse est en [1] »)
Fiches couvertes (questions avec réponse, train) : 25 / 25 ; fiches non couvertes : []
Exemples hors guide (train) par type : {'unanswerable_near': 46, 'unanswerable_off': 9}


## 9. Scan de fuite vers le test

In [ ]:
# faits-clés informatifs calculés UNE fois par question de test
INFO = {}
for r in test_rows:
    if r["category"] == "in_corpus":
        info = informative_groups(r["key_facts"], chunk_norm)
        ok = bool(info) and (len(info) >= 2 or group_rate(info[0], chunk_norm) <= 0.05)
        INFO[r["id"]] = info if ok else None

def all_facts(test_id, text):
    """Tous les faits-clés *informatifs* (≤ 15 % des chunks) de la question de test sont présents dans `text`.
    Faux s'il y a moins de 2 faits informatifs, sauf un fait très spécifique (≤ 5 % des chunks)."""
    info = INFO.get(test_id)
    if not info:
        return False
    t = norm(text)
    return all(any(rx(a).search(t) for a in alts) for alts in info)

train_ex = [ex for ex in examples_ok if ex["split"] == "train"]
leaks_ans, leaks_ctx = [], []
for r in test_rows:
    if r["category"] != "in_corpus":
        continue
    for ex in train_ex:
        if ex["kind"] == "answer" and all_facts(r["id"], ex["answer"].split("\nSources")[0]):
            leaks_ans.append((r["id"], ex["id"]))
    for ex in train_ex:
        for cid in ex["context_chunks"]:
            j = int(cid[1:])
            if all_facts(r["id"], chunks[j]["text"]):
                leaks_ctx.append((r["id"], ex["id"], cid))
print(f"Réponses d'entraînement contenant à elles seules tous les faits-clés d'une question de test : {len(set(leaks_ans))}")
print(f"Passages d'entraînement contenant tous les faits-clés d'une question de test : {len(set((a, c) for a, _, c in leaks_ctx))}")
for a, b in sorted(set(leaks_ans))[:10]: print("  réponse :", a, "<-", b)
for a, b, c in sorted(set(leaks_ctx))[:10]: print("  passage :", a, "<-", b, c)
print("\nLimite : le scan ne détecte que les faits couverts par les regex (faits-clés informatifs). Il ne remplace pas la relecture humaine.")

Réponses d'entraînement contenant à elles seules tous les faits-clés d'une question de test : 0
Passages d'entraînement contenant tous les faits-clés d'une question de test : 0

Limite : le scan ne détecte que les faits couverts par les regex (faits-clés informatifs). Il ne remplace pas la relecture humaine.


## 10. Relecture humaine 

In [ ]:
rev_rows = []
for kind, n in (("answer", 30), ("miss", 10), ("unanswerable", 10)):
    pool = [ex for ex in train_ex if ex["kind"] == kind]
    for ex in rng.sample(pool, min(n, len(pool))):
        ctx = [chunks[int(c[1:])] for c in ex["context_chunks"]]
        gold_txt = "\n---\n".join(f"[{p}] {cite(ctx[p-1])}\n{ctx[p-1]['text']}" for p in ex["gold_positions"]) if ex["gold_positions"] else "(aucun : abstention attendue)"
        rev_rows.append(
          {"id": ex["id"], "kind": ex["kind"], "question": ex["question"], "réponse_cible": ex["answer"],
                         "passage_cité": gold_txt, "contexte (extraits)": " || ".join(f"[{i+1}] " + textwrap.shorten(c['text'].replace(chr(10), ' '), 110) for i, c in enumerate(ctx)),
                         "human_ok": "", "comment": ""})
pd.DataFrame(rev_rows).to_csv(OUTPUT_DIR / "review_sample.csv", index=False)
print(f"{len(rev_rows)} lignes à relire -> {OUTPUT_DIR / 'review_sample.csv'}")
display(pd.DataFrame(rev_rows)[["id", "kind", "question", "réponse_cible"]].head(6))

47 lignes à relire -> outputs/step3_dataset/review_sample.csv


,id,kind,question,réponse_cible
0,tr061,answer,Peut-on tester avec des données réelles dans un environnement de pré-production ?,"Oui si les tests sur données fictives sont insuffisants, à condition que la pré-production soit configurée et sécurisée au même niveau q..."
1,tr112,answer,Peut-on ajuster les échelles de gravité et de vraisemblance pendant l'analyse de risques ?,"Non, c'est une mauvaise pratique : elles doivent être définies en amont, en tenant compte du contexte de l'organisme.\nSources : [4]"
2,tr145,answer,Quels acteurs identifier pour organiser l'accès aux API ?,"Les acteurs et leur rôle fonctionnel : détenteur de données, gestionnaire d'API et réutilisateur.\nSources : [4]"
3,tr107,answer,Faut-il tenir un registre des violations de données personnelles ?,"Oui, un registre interne de toutes les violations de données personnelles.\nSources : [1]"
4,tr061,answer,Peut-on tester avec des données réelles dans un environnement de pré-production ?,"Oui si les tests sur données fictives sont insuffisants, à condition que la pré-production soit configurée et sécurisée au même niveau q..."
5,tr123,answer,Quel logiciel permet de créer des conteneurs chiffrés ?,VeraCrypt ; GNU Privacy Guard permet la cryptographie asymétrique (signature et chiffrement).\nSources : [4]


## 11. Longueur en tokens et `max_seq_length` 


In [ ]:
def n_tokens(msgs, tok=None):
    if tok is not None:
        # Méthode 100% robuste : on génère le texte brut, puis on le compte classiquement
        texte_complet = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return len(tok.encode(texte_complet))

    return int(sum(len(m["content"]) for m in msgs) / 3.0)

tok = None
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    print("tokenizer Qwen chargé")
except Exception as e:
    print(f"[!] tokenizer indisponible ({type(e).__name__}) -> estimation à 3,0 car./token (prudent)")

lens = np.array([n_tokens(ex["messages"], tok) for ex in examples_ok])
print(f"longueur (tokens) : moyenne {lens.mean():.0f} | p95 {np.percentile(lens, 95):.0f} | max {lens.max()}")
MAX_SEQ_LENGTH = int(math.ceil(lens.max() / 256.0) * 256)
print(f"max_seq_length recommandé pour l'étape 4 : {MAX_SEQ_LENGTH}")

tokenizer Qwen chargé
longueur (tokens) : moyenne 1191 | p95 1376 | max 1470
max_seq_length recommandé pour l'étape 4 : 1536


## 12. Écriture du dataset

In [ ]:
def to_record(ex):
    return {"messages": ex["messages"],
            "meta": {k: ex[k] for k in ("id", "kind", "split", "variant", "fiche", "gold_chunks", "gold_positions", "context_chunks")}}

for split in ("train", "val"):
    with open(OUTPUT_DIR / f"{split}.jsonl", "w", encoding="utf-8") as f:
        for ex in examples_ok:
            if ex["split"] == split:
                f.write(json.dumps(to_record(ex), ensure_ascii=False) + "\n")

stats = {
    "n_train": int(sum(ex["split"] == "train" for ex in examples_ok)), "n_val": int(sum(ex["split"] == "val" for ex in examples_ok)),
    "train_kinds": dict(Counter(ex["kind"] for ex in examples_ok if ex["split"] == "train")),
    "val_kinds": dict(Counter(ex["kind"] for ex in examples_ok if ex["split"] == "val")),
    "abstention_rate_train": float(abst_rate), "blocked_chunks": len(BLOCKED), "n_chunks": len(chunks),
    "dropped_questions": [d[0] for d in dropped], "failed_checks": int(len(bad)),
    "leaks_answer": len(set(leaks_ans)), "leaks_passage": len(set((a, c) for a, _, c in leaks_ctx)),
    "max_seq_length": MAX_SEQ_LENGTH, "token_len_p95": float(np.percentile(lens, 95)),
}
cfg = {"step": "3_dataset", "seed": SEED, "val_fraction": VAL_FRACTION, "n_variants_train": N_VARIANTS_TRAIN, "miss_rate": MISS_RATE, "k_context": K_CONTEXT,
       "step2_config": {k: index_cfg[k] for k in ("embedder", "method", "chunking", "k_final", "n_chunks")},
       "sha256": {"seed": SEED_SHA, "test": TEST_SHA, "dev": hashlib.sha256(DEV_SET_PATH.read_bytes()).hexdigest()},
       "stats": stats, "human_review": "À COMPLÉTER : taux de lignes valides de review_sample.csv"}
json.dump(cfg, open(OUTPUT_DIR / "dataset_config.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps(stats, indent=2, ensure_ascii=False))
print("Fichiers :", sorted(p.name for p in OUTPUT_DIR.iterdir()))

{
  "n_train": 226,
  "n_val": 28,
  "train_kinds": {
    "answer": 164,
    "miss": 7,
    "unanswerable": 55
  },
  "val_kinds": {
    "answer": 16,
    "miss": 4,
    "unanswerable": 8
  },
  "abstention_rate_train": 0.2743362831858407,
  "blocked_chunks": 75,
  "n_chunks": 239,
  "dropped_questions": [
    "tr001",
    "tr005",
    "tr017",
    "tr019",
    "tr020",
    "tr021",
    "tr022",
    "tr024",
    "tr025",
    "tr032",
    "tr038",
    "tr040",
    "tr041",
    "tr042",
    "tr044",
    "tr050",
    "tr052",
    "tr053",
    "tr055",
    "tr064",
    "tr065",
    "tr066",
    "tr067",
    "tr068",
    "tr072",
    "tr082",
    "tr083",
    "tr084",
    "tr085",
    "tr086",
    "tr088",
    "tr094",
    "tr095",
    "tr096",
    "tr097",
    "tr098",
    "tr101",
    "tr109",
    "tr110",
    "tr116",
    "tr118",
    "tr119",
    "tr120",
    "tr121",
    "tr122",
    "tr127",
    "tr128",
    "tr130",
    "tr131",
    "tr142",
    "tr143",
    "tr144"
  ],
  "failed_ch

### 12.1 Exemples (à lire avant d'entraîner)

In [ ]:
for kind in ("answer", "miss", "unanswerable"):
    ex = next(e for e in train_ex if e["kind"] == kind)
    print(f"\n===== {kind.upper()} ({ex['id']}) =====")
    print(ex["messages"][1]["content"][-900:] if kind != "answer" else ex["messages"][1]["content"][:1400])
    print("\n>>> CIBLE :", ex["answer"])


===== ANSWER (tr002) =====
Passages :

[1] (Fiche 23 — Applications mobiles : conception et développement — Les précautions élémentaires, p. 53)
Minimiser les traitements de données personnelles mis en œuvre en s'assurant que chaque type de données collectées est bien nécessaire au fonctionnement de l'application.

Choisir, lors de leur sélection, les permissions pertinentes au fonctionnement de l'application et impliquant le minimum de collecte supplémentaire, voire proposer des alternatives à l'utilisateur ne reposant pas sur des permissions (ex. : la géolocalisation permet de simplifier une recherche géographique, mais peut être remplacée par la saisie manuelle de l'adresse).

Sécuriser les communications, au moins avec les serveurs, en les encapsulant dans un canal TLS, en respectant le guide TLS de l'ANSSI.

[2] (Fiche 20 — Analyse de risques — Notes de bas de page, p. 45)
[87] « Ce qu'il faut savoir sur l'analyse d'impact relative à la protection des données (AIPD) », cnil.fr
[8